# 06 全连接神经网络 (FCNN) 量化交易策略

## 论文参考

- **作者**: Jiahao Chen
- **年份**: 2023
- **标题**: *Analysis of Frequent Trading Effects of Various Machine Learning Models*
- **关键成果**: 使用多层感知机(MLP)模型对股票日收益方向进行预测，实现净利润 $31,700

### 摘要

该研究系统分析了多种机器学习模型在高频/频繁交易场景下的表现，其中全连接神经网络(FCNN)通过
多层非线性变换捕获因子间的复杂交互关系，在方向预测任务上展现出显著的盈利能力。模型采用
3层MLP结构，配合BatchNorm和Dropout正则化，有效缓解过拟合问题。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 全连接神经网络 (Fully Connected Neural Network)

FCNN（也称MLP，多层感知机）是最基础的前馈神经网络。每一层的计算如下：

$$\mathbf{h}^{(l)} = \sigma\left(\mathbf{W}^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)$$

其中 $\sigma(\cdot)$ 为激活函数（本文使用ReLU），$\mathbf{W}^{(l)}$, $\mathbf{b}^{(l)}$ 为第 $l$ 层的权重和偏置。

### 模型结构

$$\text{Input}(d) \xrightarrow{\text{FC}} 128 \xrightarrow{\text{BN+ReLU+Drop}} 64 \xrightarrow{\text{BN+ReLU+Drop}} 32 \xrightarrow{\text{FC}} 1 \xrightarrow{\text{Sigmoid}} p$$

### BatchNorm 归一化

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \gamma + \beta$$

通过对每层输入做标准化，加速收敛并起到轻微的正则化效果。

### Dropout 正则化

训练时以概率 $p=0.3$ 随机将神经元输出置零：

$$\tilde{h}_i = h_i \cdot m_i, \quad m_i \sim \text{Bernoulli}(1-p)$$

### 损失函数

二分类交叉熵损失：

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

In [ ]:
# ============================================================
# Cell 3: 动画 - 前向传播与Dropout可视化
# ============================================================
import numpy as np
import plotly.graph_objects as go

def animate_fcnn_forward():
    """动画展示FCNN前向传播过程，激活值点亮神经元，Dropout随机关闭"""
    np.random.seed(42)
    layer_sizes = [10, 128, 64, 32, 1]  # 简化显示为 [10,8,6,4,1]
    display_sizes = [10, 8, 6, 4, 1]
    layer_names = ['Input\n(因子)', 'FC-128\n+BN+ReLU', 'FC-64\n+BN+ReLU', 'FC-32\n+BN+ReLU', 'Output\n(Sigmoid)']
    x_pos = [0, 2, 4, 6, 8]
    dropout_rate = 0.3

    frames = []
    n_steps = len(layer_sizes)

    for step in range(n_steps):
        node_x, node_y, node_color, node_size, node_text = [], [], [], [], []
        edge_x, edge_y = [], []

        for li in range(len(display_sizes)):
            n = display_sizes[li]
            y_positions = np.linspace(-n/2 * 0.5, n/2 * 0.5, n)

            for ni, yp in enumerate(y_positions):
                node_x.append(x_pos[li])
                node_y.append(yp)

                if li < step:
                    # 已激活层
                    if li > 0 and li < len(display_sizes) - 1:
                        # Dropout: 随机关闭部分神经元
                        dropped = np.random.random() < dropout_rate
                        if dropped:
                            node_color.append('rgba(200,200,200,0.3)')
                            node_size.append(10)
                            node_text.append('OFF')
                        else:
                            activation = np.random.random()
                            node_color.append(f'rgba(33,150,243,{0.4 + activation * 0.6})')
                            node_size.append(14 + activation * 8)
                            node_text.append(f'{activation:.2f}')
                    else:
                        activation = np.random.random()
                        node_color.append(f'rgba(76,175,80,{0.4 + activation * 0.6})')
                        node_size.append(14 + activation * 6)
                        node_text.append(f'{activation:.2f}')
                elif li == step:
                    # 当前正在计算的层 - 高亮
                    node_color.append('rgba(255,152,0,0.9)')
                    node_size.append(18)
                    node_text.append('...')
                else:
                    # 未激活层
                    node_color.append('rgba(200,200,200,0.4)')
                    node_size.append(10)
                    node_text.append('')

            # 画连接线
            if li > 0 and li <= step:
                prev_n = display_sizes[li - 1]
                prev_y = np.linspace(-prev_n/2 * 0.5, prev_n/2 * 0.5, prev_n)
                for py in prev_y:
                    for cy in y_positions:
                        edge_x.extend([x_pos[li-1], x_pos[li], None])
                        edge_y.extend([py, cy, None])

        data = [
            go.Scatter(x=edge_x, y=edge_y, mode='lines',
                       line=dict(color='rgba(150,150,150,0.15)', width=0.5),
                       hoverinfo='none', showlegend=False),
            go.Scatter(x=node_x, y=node_y, mode='markers+text',
                       marker=dict(color=node_color, size=node_size,
                                   line=dict(color='white', width=1)),
                       text=node_text, textposition='middle center',
                       textfont=dict(size=7, color='white'),
                       hoverinfo='text', showlegend=False),
        ]
        # 层标签
        for li in range(len(display_sizes)):
            n = display_sizes[li]
            data.append(go.Scatter(
                x=[x_pos[li]], y=[n/2 * 0.5 + 0.6],
                mode='text', text=[layer_names[li]],
                textfont=dict(size=10, color='#333'),
                showlegend=False, hoverinfo='none'
            ))

        titles = ['输入因子向量', '第1隐藏层: FC(128)+BN+ReLU+Dropout',
                  '第2隐藏层: FC(64)+BN+ReLU+Dropout',
                  '第3隐藏层: FC(32)+BN+ReLU+Dropout', '输出层: Sigmoid']
        frames.append(go.Frame(data=data, name=titles[step]))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=dict(text='FCNN 前向传播动画 (激活点亮 + Dropout关闭神经元)'),
        xaxis=dict(visible=False, range=[-1, 9]),
        yaxis=dict(visible=False, range=[-3.5, 4]),
        height=500, width=900,
        plot_bgcolor='white',
        updatemenus=[dict(type='buttons', x=0.1, y=1.12, buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 1200}, 'transition': {'duration': 400}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0, x=0.05, len=0.9,
        )],
    )
    return fig

fig = animate_fcnn_forward()
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与环境设置
# ============================================================
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# 添加项目根目录到路径
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_stock_daily
from shared.backtest_engine import Backtester
from shared.visualization import plot_full_report, set_chinese_font
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands,
    price_to_ma, atr, volume_price_corr, high_low_range, turnover_factor
)

set_chinese_font()

# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
SYMBOL = '600519'  # 贵州茅台
df = get_stock_daily(SYMBOL, start_date='20200601', end_date='20241231')
print(f'数据形状: {df.shape}')
print(f'日期范围: {df.index[0]} ~ {df.index[-1]}')
df.tail(3)

In [ ]:
# ============================================================
# Cell 6: 特征工程
# ============================================================
def build_features(df):
    """构建FCNN输入特征 - 当日因子向量（非序列）"""
    feat = pd.DataFrame(index=df.index)

    # 动量因子
    mom = momentum(df['close'], periods=[1, 3, 5, 10, 20, 60])
    feat = pd.concat([feat, mom], axis=1)

    # 波动率因子
    vol = volatility(df['close'], windows=[5, 10, 20, 60])
    feat = pd.concat([feat, vol], axis=1)

    # RSI
    feat['rsi_14'] = rsi(df['close'], 14)
    feat['rsi_7'] = rsi(df['close'], 7)

    # MACD
    macd_df = macd(df['close'])
    feat['macd_hist'] = macd_df['histogram']
    feat['macd_line'] = macd_df['macd']

    # 布林带
    bb = bollinger_bands(df['close'])
    feat['bb_pctb'] = bb['bb_pctb']
    feat['bb_width'] = bb['bb_width']

    # 均线偏离
    ptm = price_to_ma(df['close'], windows=[5, 10, 20, 60])
    feat = pd.concat([feat, ptm], axis=1)

    # ATR
    feat['atr_14'] = atr(df['high'], df['low'], df['close'], 14)

    # 量价相关
    feat['vp_corr'] = volume_price_corr(df['close'], df['volume'], 20)

    # 日内波幅
    feat['hl_range'] = high_low_range(df['high'], df['low'], df['close'])

    # 换手率
    if 'turnover' in df.columns:
        tf = turnover_factor(df['turnover'], windows=[5, 10, 20])
        feat = pd.concat([feat, tf], axis=1)

    # 成交量变化率
    feat['vol_change_5'] = df['volume'].pct_change(5)
    feat['vol_change_10'] = df['volume'].pct_change(10)

    # 标签: 次日涨跌方向 (1=涨, 0=跌)
    feat['target'] = (df['close'].shift(-1) > df['close']).astype(int)

    feat.dropna(inplace=True)
    return feat

features = build_features(df)
print(f'特征矩阵形状: {features.shape}')
print(f'特征列: {[c for c in features.columns if c != "target"]}')
print(f'\n标签分布:\n{features["target"].value_counts()}')

In [ ]:
# ============================================================
# Cell 7: PyTorch 模型定义与训练
# ============================================================

# --- 模型定义 ---
class FCNN(nn.Module):
    """3层MLP: input->128->64->32->1, 配合BatchNorm + Dropout(0.3) + ReLU"""
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# --- 数据准备 ---
TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'

feature_cols = [c for c in features.columns if c != 'target']
train_data = features.loc[:TRAIN_END]
test_data = features.loc[TEST_START:]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_data[feature_cols].values)
y_train = train_data['target'].values.astype(np.float32)
X_test = scaler.transform(test_data[feature_cols].values)
y_test = test_data['target'].values.astype(np.float32)

print(f'训练集: {X_train.shape[0]} 样本, {X_train.shape[1]} 特征')
print(f'测试集: {X_test.shape[0]} 样本')

train_dataset = TensorDataset(
    torch.FloatTensor(X_train),
    torch.FloatTensor(y_train)
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 训练 ---
model = FCNN(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

EPOCHS = 60
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(y_batch)
    scheduler.step()

    avg_loss = epoch_loss / len(train_dataset)
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}  LR: {scheduler.get_last_lr()[0]:.6f}')

# --- 预测 ---
model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test).to(device)
    test_probs = model(X_test_t).cpu().numpy()

test_preds = (test_probs > 0.5).astype(int)
accuracy = (test_preds == y_test).mean()
print(f'\n测试集准确率: {accuracy:.4f}')
print(f'预测分布: 涨={test_preds.sum()}, 跌={len(test_preds) - test_preds.sum()}')

In [ ]:
# ============================================================
# Cell 8: 回测
# ============================================================
# 构建信号: 预测涨→做多(1), 预测跌→空仓(0)
signals = pd.Series(test_preds, index=test_data.index, name='signal')
signals = signals.replace(0, 0)  # 0=空仓

# 获取测试期间价格
test_prices = df.loc[test_data.index, 'close']

# 执行回测
bt = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = bt.run(test_prices, signals)

print('回测绩效指标:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 训练损失曲线
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color='#2196F3', linewidth=1.5)
ax.set_title('FCNN 训练损失曲线', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 完整回测报告
plot_full_report(result, strategy_name='FCNN (Chen 2023)')

# 3. 预测概率分布
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(test_probs, bins=50, color='#2196F3', alpha=0.7, edgecolor='white')
ax.axvline(x=0.5, color='red', linestyle='--', label='阈值 0.5')
ax.set_title('FCNN 预测概率分布', fontsize=14, fontweight='bold')
ax.set_xlabel('预测概率 P(涨)')
ax.set_ylabel('频次')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 结果分析

### 模型表现

- **模型结构**: Input → FC(128) → BN → ReLU → Dropout(0.3) → FC(64) → BN → ReLU → Dropout(0.3) → FC(32) → BN → ReLU → Dropout(0.3) → FC(1) → Sigmoid
- **训练期**: 2021-01 至 2023-12 (约700+交易日)
- **测试期**: 2024-01 至 2024-12
- **输入**: 当日因子向量 (动量、波动率、RSI、MACD、布林带等约30维特征)

### 关键发现

1. **BatchNorm + Dropout** 的组合有效防止了过拟合，训练损失平稳下降
2. FCNN将当日所有因子作为flat向量输入，不考虑时序结构，适合因子间交互关系的捕捉
3. 该方法简单易实现，可作为深度学习量化策略的baseline

### 局限性

- 不考虑时序依赖关系，无法捕捉趋势动量等时序模式
- 对特征工程质量高度依赖
- 日频交易受T+1限制，实际换手成本较高